# 00 - GCP setup
**What it does:** <in your own words, one or two sentences>
**Needs:** a Google Cloud project with billing, and the Colab Secrets TENNIS_PROJECT_ID and TENNIS_BUCKET_NAME
**Produces:** a private bucket with protections, and the BigQuery datasets
**Run after:** nothing, this is the first notebook

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Enable the services used (Storage and BigQuery)

In [ ]:
!gcloud services enable storage.googleapis.com bigquery.googleapis.com

In [ ]:
!gcloud services list --enabled | grep -E "storage|bigquery"

## Bucket: is the name available?

In [ ]:
from google.cloud import storage
from google.api_core.exceptions import Forbidden, Conflict

client = storage.Client(project=PROJECT_ID)

try:
    existing = client.lookup_bucket(BUCKET_NAME)
    if existing:
        print("You already own this bucket. Skip the next cell.")
    else:
        print("Name appears available.")
except Forbidden:
    print("Someone else has this name. Change TENNIS_BUCKET_NAME in Secrets, re-run the settings cell, then this cell.")

## Create the bucket (private, no public access)

In [ ]:
bucket = storage.Bucket(client, BUCKET_NAME)
bucket.iam_configuration.uniform_bucket_level_access_enabled = True
bucket.iam_configuration.public_access_prevention = "enforced"

try:
    client.create_bucket(bucket, location=REGION)
    print("Created bucket in", REGION)
except Conflict as e:
    print("Not created:", e)

## Set object versioning and the lifecycle rule

In [ ]:
b = client.get_bucket(BUCKET_NAME)      # edit the copy fetched from Google, then patch that same copy

b.versioning_enabled = True             # keep old versions of overwritten or deleted files
b.clear_lifecycle_rules()               # replaces any existing rules, so re-running adds no duplicates
b.add_lifecycle_delete_rule(is_live=False, number_of_newer_versions=3)   # old versions go once 3+ newer ones exist
b.patch()
print("Protections updated.")

## Check the bucket settings by reading them back from Google

In [ ]:
b = client.get_bucket(BUCKET_NAME)
print("location:", b.location)
print("versioning:", b.versioning_enabled)
print("lifecycle rules:", list(b.lifecycle_rules))        # one Delete rule with numNewerVersions 3
print("public access prevention:", b.iam_configuration.public_access_prevention)
print("uniform access:", b.iam_configuration.uniform_bucket_level_access_enabled)

sd = getattr(b, "soft_delete_policy", None)
if sd is not None:
    print("soft delete (days):", (sd.retention_duration_seconds or 0) / 86400)
else:
    print("soft delete: not shown by this library version; check the bucket's Protection tab")

## Create the BigQuery datasets (raw, clean, features)

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

for name in DATASETS:
    dataset = bigquery.Dataset(f"{PROJECT_ID}.{name}")
    dataset.location = REGION
    dataset.labels = {"project": "tennis-ml"}
    bq.create_dataset(dataset, exists_ok=True)

for d in bq.list_datasets():
    full = bq.get_dataset(d.reference)
    print(full.dataset_id, "|", full.location)